In [33]:
import os
import re
import ast
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.append('..')

from matplotlib.ticker import SymmetricalLogLocator, LogFormatterSciNotation
from matplotlib.ticker import AutoMinorLocator

DEFAULT_FIGSIZE = (5, 4)
DEFAULT_FONT_SIZE = 10


In [34]:
def generate_density(eigenvalues, weights, num_grid_points=int(1e5), sigma_squared=1e-5, boundary_margin=1e-3):

  def gaussian(x, x0, variance):
    return np.exp(-(x0 - x)**2 / (2.0 * variance)) / np.sqrt(2 * np.pi * variance)

  eigenvalues = np.array(eigenvalues).real
  weights = np.array(weights).real

  num_runs, num_eigvals = eigenvalues.shape

  lambda_max = np.mean(np.max(eigenvalues, axis=1), axis=0) + boundary_margin
  lambda_min = np.mean(np.min(eigenvalues, axis=1), axis=0) - boundary_margin

  grid_points = np.linspace(lambda_min, lambda_max, num=num_grid_points)
  grid_length = (lambda_max - lambda_min) / (num_grid_points - 1)

  sigma = sigma_squared * max(1, (lambda_max - lambda_min))

  # compute density
  densities = np.zeros((num_runs, num_grid_points), dtype=eigenvalues.dtype)
  for i in range(num_runs):
    for j in range(num_grid_points):
      x = grid_points[j]
      convolutions = gaussian(eigenvalues[i,:], x, sigma)
      densities[i,j] = np.sum(convolutions * weights[i,:])

  # average across runs
  densities = np.mean(densities, axis=0)
  densities = densities / (np.sum(densities) * grid_length)

  return densities, grid_points

In [35]:
# Utilities to parse Hessian density dumps generated in hessian_analysis/density
def parse_pyhessian_density_result(result_path):
    result_path = Path(result_path)
    text = result_path.read_text()
    eigen_match = re.search(r'eigen_list_full:\s*(\[\[.*?\]\])\s*weight_list_full', text, re.S)
    weight_match = re.search(r'weight_list_full:\s*(\[\[.*?\]\])\s*all_eigenvals', text, re.S)
    if not eigen_match or not weight_match:
        raise ValueError(f'Could not parse eigen/weight lists from {result_path}')
    eigenvalues = np.array(ast.literal_eval(eigen_match.group(1)))
    weights = np.array(ast.literal_eval(weight_match.group(1)))
    if eigenvalues.shape != weights.shape:
        raise ValueError(f'Mismatched eigen/weight shapes in {result_path}: {eigenvalues.shape} vs {weights.shape}')
    return eigenvalues, weights

def extract_setting_metadata(setting_name):
    metadata = {}
    k_match = re.search(r'k([0-9.]+)_([0-9.]+)', setting_name)
    if k_match:
        metadata['k_range'] = (float(k_match.group(1)), float(k_match.group(2)))
    val_match = re.search(r'val(\d+)', setting_name)
    if val_match:
        metadata['val_points'] = int(val_match.group(1))
    return metadata

def load_hessian_density_results(base_dir='hessian_analysis/density/hessian_analysis'):
    base_path = Path(base_dir)
    if not base_path.exists():
        raise FileNotFoundError(f'Cannot find {base_path} from the current working directory')

    experiments = []
    for result_path in sorted(base_path.glob('**/hessian_pyhessian_density_results.txt')):
        eigenvalues, weights = parse_pyhessian_density_result(result_path)
        relative = result_path.relative_to(base_path)
        eps_dir, setting, _, bsz_lr, seed = relative.parts[:5]
        experiment = {
            'eps_dir': eps_dir,
            'setting': setting,
            'bsz_lr': bsz_lr,
            'seed': seed,
            'result_path': result_path,
            'eigenvalues': eigenvalues,
            'weights': weights,
        }
        if eps_dir.startswith('expts_eps'):
            experiment['eps'] = eps_dir.replace('expts_eps', '')
        experiment.update(extract_setting_metadata(setting))
        experiments.append(experiment)

    return experiments

In [36]:
# Plotting helpers for the pyhessian density text dumps
def format_density_label(experiment):
    k_range = experiment.get('k_range')
    parts = [experiment.get('eps_dir'), experiment.get('setting')]
    if k_range:
        parts.append(f"k=[{k_range[0]}, {k_range[1]}]")
    if experiment.get('val_points') is not None:
        parts.append(f"val{experiment.get('val_points')}")
    parts.append(experiment.get('bsz_lr'))
    parts.append(experiment.get('seed'))
    return ' | '.join([p for p in parts if p])

def compute_symlog_xlim(grid, density, y_cutoff, padding_ratio):
    indices = np.argwhere(density >= y_cutoff)
    if indices.size == 0:
        return grid[0], grid[-1]
    xlim_lower = grid[indices[0][0]]
    xlim_upper = grid[indices[-1][0]]
    lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
    xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
    if (xlim_lower <= 0) and (xlim_lower > -0.5):
        xlim_lower = -0.5
    upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
    xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
    xlim_upper = xlim_upper * (1 + padding_ratio)
    return xlim_lower, xlim_upper

def plot_hessian_density_experiment(experiment, gaussian_variance=5e-5, y_cutoff=1e-10,
                                    padding_ratio=0.1, fixed_linthresh=1.0,
                                    output_dir='spectral_density_plots/hessian_analysis',
                                    show_plot=False, line_color='tab:blue',
                                    font_size=DEFAULT_FONT_SIZE, show_title=False,
                                    figsize=DEFAULT_FIGSIZE):
    densities, grid = generate_density(experiment['eigenvalues'], experiment['weights'],
                                       sigma_squared=gaussian_variance)
    xlim_lower, xlim_upper = compute_symlog_xlim(grid, densities, y_cutoff, padding_ratio)

    mpl.rcParams.update({'font.size': font_size})
    fig, ax = plt.subplots(figsize=figsize)
    ax.semilogy(grid, densities, color=line_color)
    ax.set_xscale('symlog', linthresh=fixed_linthresh)
    ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
    ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))
    ax.set_ylim(bottom=y_cutoff)
    ax.set_xlim(left=xlim_lower, right=xlim_upper)
    ax.set_ylabel('Density')   #, fontname='Times New Roman')
    ax.set_xlabel('Eigenvalue')   #, fontname='Times New Roman')
    if show_title:
        ax.set_title(format_density_label(experiment))

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    filename = f"density_{experiment['eps_dir']}_{experiment['setting']}_{experiment['bsz_lr']}_{experiment['seed']}.pdf"
    output_path = output_dir / filename
    fig.savefig(output_path, bbox_inches='tight')

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return output_path

def plot_all_hessian_densities(experiments, **kwargs):
    saved_paths = []
    for experiment in experiments:
        saved_paths.append(plot_hessian_density_experiment(experiment, **kwargs))
    return saved_paths

def make_density_key(experiment):
    return (
        experiment.get('eps_dir'),
        experiment.get('setting'),
        experiment.get('bsz_lr'),
        experiment.get('seed'),
    )


def match_density_experiments(before_experiments, after_experiments):
    before_index = {make_density_key(exp): exp for exp in before_experiments}
    after_index = {make_density_key(exp): exp for exp in after_experiments}

    matched = []
    for key in sorted(before_index):
        if key in after_index:
            matched.append({'before': before_index[key], 'after': after_index[key]})

    missing_after = sorted(before_index.keys() - after_index.keys())
    missing_before = sorted(after_index.keys() - before_index.keys())
    return matched, missing_after, missing_before


def compute_combined_symlog_xlim(before_grid, before_density, after_grid, after_density,
                                 y_cutoff, padding_ratio):
    indices_before = np.argwhere(before_density >= y_cutoff)
    indices_after = np.argwhere(after_density >= y_cutoff)

    if indices_before.size == 0 and indices_after.size == 0:
        xlim_lower = min(before_grid[0], after_grid[0])
        xlim_upper = max(before_grid[-1], after_grid[-1])
    else:
        candidates = []
        if indices_before.size:
            candidates.append((before_grid[indices_before[0][0]], before_grid[indices_before[-1][0]]))
        if indices_after.size:
            candidates.append((after_grid[indices_after[0][0]], after_grid[indices_after[-1][0]]))
        xlim_lower = min(c[0] for c in candidates)
        xlim_upper = max(c[1] for c in candidates)

    lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
    xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
    if (xlim_lower <= 0) and (xlim_lower > -0.5):
        xlim_lower = -0.5
    upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
    xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
    xlim_upper = xlim_upper * (1 + padding_ratio)
    return xlim_lower, xlim_upper


def plot_hessian_density_comparison(before_experiment, after_experiment, gaussian_variance=5e-5,
                                    y_cutoff=1e-10, padding_ratio=0.1, fixed_linthresh=1.0,
                                    output_dir='spectral_density_plots/hessian_analysis_before_after',
                                    show_plot=False, line_colors=None, line_styles=None,
                                    font_size=DEFAULT_FONT_SIZE, show_title=False,
                                    figsize=DEFAULT_FIGSIZE):
    if line_colors is None:
        line_colors = {'hessian_begin': 'tab:blue', 'hessian_final': 'tab:orange'}
    if line_styles is None:
        line_styles = {'hessian_begin': 'solid', 'hessian_final': 'dashed'}

    before_density, before_grid = generate_density(
        before_experiment['eigenvalues'], before_experiment['weights'],
        sigma_squared=gaussian_variance
    )
    after_density, after_grid = generate_density(
        after_experiment['eigenvalues'], after_experiment['weights'],
        sigma_squared=gaussian_variance
    )

    xlim_lower, xlim_upper = compute_combined_symlog_xlim(
        before_grid, before_density, after_grid, after_density, y_cutoff, padding_ratio
    )

    mpl.rcParams.update({'font.size': font_size})
    fig, ax = plt.subplots(figsize=figsize)
    ax.semilogy(
        before_grid, before_density,
        color=line_colors['hessian_begin'], linestyle=line_styles['hessian_begin']
    )
    ax.semilogy(
        after_grid, after_density,
        color=line_colors['hessian_final'], linestyle=line_styles['hessian_final']
    )

    ax.set_xscale('symlog', linthresh=fixed_linthresh)
    ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
    ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))
    ax.set_ylim(bottom=y_cutoff)
    ax.set_xlim(left=xlim_lower, right=xlim_upper)
    ax.set_ylabel('Density')   #, fontname='Times New Roman')
    ax.set_xlabel('Eigenvalue')   #, fontname='Times New Roman')
    if show_title:
        ax.set_title(format_density_label(after_experiment))

    legend_elements = [
        plt.Line2D([0], [0], color=line_colors['hessian_begin'],
                   linestyle=line_styles['hessian_begin'], label='Before Training'),
        plt.Line2D([0], [0], color=line_colors['hessian_final'],
                   linestyle=line_styles['hessian_final'], label='After Training')
    ]
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=2)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    filename = (
        f"density_{after_experiment['setting']}"
        f"_{after_experiment['bsz_lr']}.pdf"
    )
    output_path = output_dir / filename
    fig.savefig(output_path, bbox_inches='tight')

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return output_path


def plot_all_hessian_density_comparisons(paired_experiments, **kwargs):
    saved_paths = []
    for pair in paired_experiments:
        saved_paths.append(
            plot_hessian_density_comparison(pair['before'], pair['after'], **kwargs)
        )
    return saved_paths


In [37]:
def get_title(system, Nf, pde_params):
    if system == 'convection':
        return rf'Convection, $n_{{\rm res}} = {Nf}, \beta = {pde_params}$'

In [38]:
base_dir = Path('hessian_analysis/density/hessian_analysis')
density_experiments = load_hessian_density_results(base_dir)
print(f'Loaded {len(density_experiments)} Hessian density files from {base_dir}')

# Example filter: keep only eps150
# density_experiments = [exp for exp in density_experiments if exp['eps_dir'] == 'expts_eps150']


Loaded 16 Hessian density files from hessian_analysis/density/hessian_analysis


In [39]:
init_base_dir = Path('hessian_analysis/density_init/hessian_analysis')
density_init_experiments = load_hessian_density_results(init_base_dir)
print(f'Loaded {len(density_init_experiments)} Hessian init density files from {init_base_dir}')

paired_density_experiments, missing_after, missing_before = match_density_experiments(
    density_init_experiments, density_experiments
)
print(f'Matched {len(paired_density_experiments)} init/train density pairs')
print(f'Init only: {len(missing_after)} | Train only: {len(missing_before)}')


Loaded 16 Hessian init density files from hessian_analysis/density_init/hessian_analysis
Matched 16 init/train density pairs
Init only: 0 | Train only: 0


In [40]:
def plot_spectral_density(pdes, betas, Nf_list, saved_results,
                          line_colors, line_styles, font_size=DEFAULT_FONT_SIZE,
                          gaussian_variance=5e-5, y_cutoff=1e-10,
                          folder_path='spectral_density_plots', filename='spectral_density',
                          show_plot=True, padding_ratio=0.1, fixed_linthresh=1,
                          figsize=DEFAULT_FIGSIZE):
    mpl.rcParams.update({'font.size': font_size})

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    legend_elements = [
        plt.Line2D([0], [0], color=line_colors['hessian_begin'], linestyle=line_styles['hessian_begin'], label='Before Training'),
        plt.Line2D([0], [0], color=line_colors['hessian_final'], linestyle=line_styles['hessian_final'], label='After Training')
    ]

    for pde in pdes:
        for beta in betas:
            for Nf in Nf_list:
                pde_result = saved_results[pde][f'({beta}, {Nf})']

                hessian_densities_before, hessian_grid_before = generate_density(
                    pde_result["eigen_begin"],
                    pde_result["weight_begin"],
                    sigma_squared=gaussian_variance
                )
                hessian_densities_final, hessian_grid_final = generate_density(
                    pde_result["eigen_final"],
                    pde_result["weight_final"],
                    sigma_squared=gaussian_variance
                )

                xlim_idx_hessian = np.argwhere(hessian_densities_before >= y_cutoff)[[0, -1]]
                xlim_idx_final = np.argwhere(hessian_densities_final >= y_cutoff)[[0, -1]]

                xlim_lower = min(hessian_grid_before[xlim_idx_hessian[0]], hessian_grid_final[xlim_idx_final[0]])
                xlim_upper = max(hessian_grid_before[xlim_idx_hessian[1]], hessian_grid_final[xlim_idx_final[1]])

                lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
                xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
                if (xlim_lower <= 0) & (xlim_lower > -0.5):
                    xlim_lower = -0.5

                upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
                xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
                xlim_upper = xlim_upper * (1 + padding_ratio)

                fig, ax = plt.subplots(figsize=figsize)
                ax.semilogy(hessian_grid_before, hessian_densities_before,
                            color=line_colors['hessian_begin'], linestyle=line_styles['hessian_begin'])
                ax.semilogy(hessian_grid_final, hessian_densities_final,
                            color=line_colors['hessian_final'], linestyle=line_styles['hessian_final'])

                ax.set_xscale('symlog', linthresh=fixed_linthresh)
                ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
                ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))

                ax.set_ylim(bottom=y_cutoff)
                ax.set_xlim(left=xlim_lower, right=xlim_upper)
                ax.set_ylabel('Density')   #, fontname='Times New Roman')
                ax.set_xlabel('Eigenvalue')   #, fontname='Times New Roman')
                # ax.set_title(get_title(pde, Nf, beta))

                fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=2)
                fig.tight_layout(rect=[0, 0.03, 1, 0.95])

                full_filename = f"{filename}_{pde}_Nf_{Nf}_beta_{beta}.pdf"
                fig.savefig(os.path.join(folder_path, full_filename), bbox_inches='tight')

                if show_plot:
                    plt.show()
                else:
                    plt.close(fig)


In [41]:
# saved_paths = plot_all_hessian_densities(
#     density_experiments,
#     gaussian_variance=5e-5,
#     y_cutoff=1e-10,
#     padding_ratio=0.1,
#     fixed_linthresh=1.0,
#     output_dir='spectral_density_plots/hessian_analysis',
#     show_plot=False
# )
# print(f'Saved {len(saved_paths)} plots to spectral_density_plots/hessian_analysis')

In [42]:
comparison_paths = plot_all_hessian_density_comparisons(
    paired_density_experiments,
    gaussian_variance=5e-5,
    y_cutoff=1e-10,
    padding_ratio=0.1,
    fixed_linthresh=1.0,
    output_dir='spectral_density_plots/hessian_analysis_before_after',
    show_plot=False
)
print('Saved {0} before/after plots to spectral_density_plots/hessian_analysis_before_after'.format(
    len(comparison_paths)
))


Saved 16 before/after plots to spectral_density_plots/hessian_analysis_before_after


In [43]:
# def plot_spectral_density_final(pdes, betas, Nf_list, saved_results,
#                                     line_colors, line_styles, font_size=DEFAULT_FONT_SIZE,
#                                     gaussian_variance=5e-5, y_cutoff=1e-10,
#                                     folder_path='spectral_density_plots', filename='spectral_density_final',
#                                     show_plot=True, padding_ratio=0.1, fixed_linthresh=1,
#                                     show_title=False, figsize=DEFAULT_FIGSIZE):
#     mpl.rcParams.update({'font.size': font_size})

#     if not os.path.exists(folder_path):
#         os.makedirs(folder_path)

#     if line_colors is None:
#         line_colors = {'hessian_final': 'tab:blue'}
#     if line_styles is None:
#         line_styles = {'hessian_final': 'solid'}

#     legend_elements = [
#         plt.Line2D([0], [0], color=line_colors.get('hessian_final', 'tab:blue'),
#                    linestyle=line_styles.get('hessian_final', 'solid'), label='After Training')
#     ]

#     for pde in pdes:
#         for beta in betas:
#             for Nf in Nf_list:
#                 pde_result = saved_results[pde][f'({beta}, {Nf})']

#                 hessian_densities_final, hessian_grid_final = generate_density(
#                     pde_result["eigen_final"],
#                     pde_result["weight_final"],
#                     sigma_squared=gaussian_variance
#                 )

#                 xlim_idx_final = np.argwhere(hessian_densities_final >= y_cutoff)[[0, -1]]
#                 xlim_lower = hessian_grid_final[xlim_idx_final[0]]
#                 xlim_upper = hessian_grid_final[xlim_idx_final[1]]

#                 lower_exponent = np.floor(np.log10(np.abs(xlim_lower))) if xlim_lower != 0 else 0
#                 xlim_lower = -1 * (10 ** (lower_exponent + 1)) if xlim_lower < 0 else 10 ** lower_exponent
#                 if (xlim_lower <= 0) & (xlim_lower > -0.5):
#                     xlim_lower = -0.5

#                 upper_exponent = np.floor(np.log10(np.abs(xlim_upper))) if xlim_upper != 0 else 0
#                 xlim_upper = np.ceil(xlim_upper / (10 ** upper_exponent)) * (10 ** upper_exponent)
#                 xlim_upper = xlim_upper * (1 + padding_ratio)

#                 fig, ax = plt.subplots(figsize=figsize)
#                 ax.semilogy(hessian_grid_final, hessian_densities_final,
#                             color=line_colors.get('hessian_final', 'tab:blue'),
#                             linestyle=line_styles.get('hessian_final', 'solid'))

#                 ax.set_xscale('symlog', linthresh=fixed_linthresh)
#                 ax.xaxis.set_major_locator(SymmetricalLogLocator(base=10, linthresh=fixed_linthresh))
#                 ax.xaxis.set_major_formatter(LogFormatterSciNotation(base=10, linthresh=fixed_linthresh))

#                 ax.set_ylim(bottom=y_cutoff)
#                 ax.set_xlim(left=xlim_lower, right=xlim_upper)
#                 ax.set_ylabel('Density')   #, fontname='Times New Roman')
#                 ax.set_xlabel('Eigenvalue')   #, fontname='Times New Roman')
#                 if show_title:
#                     ax.set_title(get_title(pde, Nf, beta))

#                 fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05))
#                 fig.tight_layout(rect=[0, 0.03, 1, 0.95])

#                 full_filename = f"{filename}_{pde}_Nf_{Nf}_beta_{beta}.pdf"
#                 fig.savefig(os.path.join(folder_path, full_filename), bbox_inches='tight')

#                 if show_plot:
#                     plt.show()
#                 else:
#                     plt.close(fig)


In [44]:
# Optional: preview a single experiment interactively
# if density_experiments:
#     plot_hessian_density_experiment(density_experiments[0], show_plot=True)
